In [22]:
DOWNLOAD_DATA = False
if DOWNLOAD_DATA:
    import kagglehub

    # Download latest version
    path = kagglehub.competition_download('week-03-hidden-pairs')
    print("Path to competition files:", path)

In [3]:
import pandas as pd
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier, early_stopping
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from pathlib import Path
import time
# import cleanlab
import json
from datetime import datetime

import joblib
from importlib.metadata import version

PACKAGE_VERSIONS = {name: version(name) for name in [
    "numpy", "pandas", "scikit-learn", "lightgbm", "joblib", "torch",
]}
print(PACKAGE_VERSIONS)
from dataclasses import replace

{'numpy': '2.5.3', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'joblib': '1.6.0', 'torch': '2.14.0'}


In [4]:
def set_seed(seed=42):
    """Фиксирует seed для random, NumPy и PyTorch, включая CUDA."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f'Seed: {seed}')

In [5]:
SEED = 0xFACED
set_seed(SEED)

LOCAL = not Path('/kaggle/input').exists()

LOG_DIR = Path('./logs') if LOCAL else Path('/kaggle/working/logs')

OUTPUT_DIR = Path('.') if LOCAL else Path('/kaggle/working')
RUN_SEARCH = False
RUN_CLEANLAB = False
EARLY_STOPPING_ROUNDS = 500  # Длинное плато на validation возможно перед новым ростом.

Seed: 1027309


In [6]:
DATA_DIR = Path(path) if DOWNLOAD_DATA else (
    Path('./data') if LOCAL else Path('/kaggle/input/competitions/week-03-hidden-pairs')
)

train_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'train.csv')
val_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'validation.csv')
test_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission: pd.DataFrame = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("sample submission:", sample_submission.shape)

train: (2520, 514)
val: (560, 514)
test: (1116, 513)
sample submission: (1116, 2)


In [7]:
feature_columns: list[str] = [
    col for col in train_df.columns if col not in ['row_id', 'target']
]

assert feature_columns == [c for c in val_df if c not in ['row_id', 'target']]
assert feature_columns == [c for c in test_df if c != 'row_id']
for frame in [train_df, val_df, test_df]:
    assert np.isfinite(frame[feature_columns].to_numpy()).all(), 'Признаки содержат NaN/inf'

X_train: pd.DataFrame = train_df[feature_columns]
y_train: pd.Series = train_df['target']

X_val: pd.DataFrame = val_df[feature_columns]
y_val: pd.Series = val_df['target']

X_test: pd.DataFrame = test_df[feature_columns]

print('# of features:', len(feature_columns))
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)
print('X_test shape:', X_test.shape)

# of features: 512
X_train shape: (2520, 512)
y_train shape: (2520,)
X_val shape: (560, 512)
y_val shape: (560,)
X_test shape: (1116, 512)


In [8]:
def save_model(model, model_path):
    """Сохраняет обученную модель; возвращает путь к файлу."""
    model_path = Path(model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_path, compress=3)
    print(f'Обученная модель сохранена: {model_path}', flush=True)
    return model_path


def load_model(json_path, model_name):
    """Загружает модель по JSON запуска и точному имени из поля Model."""
    json_path = Path(json_path)
    run_log = json.loads(json_path.read_text(encoding='utf-8'))
    matches = [row for row in run_log['results'] if row['Model'] == model_name]
    if len(matches) != 1:
        available = [row['Model'] for row in run_log['results']]
        raise ValueError(f'Нужна ровно одна модель {model_name!r}. Доступны: {available}')
    model_path = json_path.parent / matches[0]['Model File']
    print(f'Загружаю {model_name}: {model_path}', flush=True)
    return joblib.load(model_path)


def save_submission(selected_model, X_test, test_ids, sample_submission, output_path):
    """Предсказывает без fit, сохраняет CSV в порядке sample_submission и возвращает его."""
    test_ids = pd.Series(test_ids).reset_index(drop=True)
    if len(X_test) != len(test_ids) or not test_ids.is_unique:
        raise ValueError('Нужен один уникальный row_id на каждую строку X_test')
    if not sample_submission['row_id'].is_unique:
        raise ValueError('В sample_submission есть повторяющиеся row_id')
    if set(test_ids) != set(sample_submission['row_id']):
        raise ValueError('Наборы row_id в test и sample_submission различаются')
    expected = getattr(selected_model, 'feature_names_in_', None)
    if expected is not None and list(X_test.columns) != list(expected):
        raise ValueError('Состав или порядок признаков X_test отличается от обучения')
    # Только предсказания: повторное обучение не запускается.
    print(f'Предсказываю для {len(X_test)} строк...', flush=True)
    test_probs = selected_model.predict_proba(X_test)[:, 1]
    if not np.isfinite(test_probs).all() or not ((test_probs >= 0) & (test_probs <= 1)).all():
        raise ValueError('Модель вернула некорректные вероятности')
    submission = sample_submission[['row_id']].merge(
        pd.DataFrame({'row_id': test_ids, 'target': test_probs}),
        on='row_id', how='left', sort=False, validate='one_to_one',
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)
    print(f'Сохранено: {output_path.resolve()} | {submission.shape}', flush=True)
    return submission

In [9]:
def _save_trained_model(model, log_path, model_index, use_cleanlab):
    """Сохраняет обученный классификатор и возвращает путь относительно LOG_DIR.

    После joblib.load() модель готова к predict_proba() без повторного обучения.
    Для CleanLearning сохраняется его финальный обученный классификатор.
    """
    model_dir = log_path.with_suffix('')
    model_dir.mkdir(parents=True, exist_ok=True)
    variant = 'cleanlab' if use_cleanlab else 'base'
    model_path = model_dir / f'{model_index:02d}_{variant}.joblib'
    save_model(model, model_path)
    return str(model_path.relative_to(log_path.parent))

def _save_pipeline_log(log_path, started_at, metrics_list):
    """Сохраняет время запуска, параметры и метрики уже обученных моделей в JSON."""
    log_data = {
        'run_started_at': started_at.isoformat(),
        'package_versions': PACKAGE_VERSIONS,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
        'updated_at': datetime.now().astimezone().isoformat(),
        'results': json.loads(pd.DataFrame(metrics_list).to_json(orient='records', double_precision=15)),
    }
    log_path.write_text(json.dumps(log_data, ensure_ascii=False, indent=2), encoding='utf-8')

def _record_result(result, run_name, cleanlab_enabled, log_path, model_index,
                   val_predictions, histories, metrics_list, extra_metrics=None):
    """Сохраняет модель и добавляет единообразную строку метрик."""
    model_file = _save_trained_model(result['model'], log_path, model_index, cleanlab_enabled)
    val_predictions[run_name] = result['val_probs']
    histories[run_name] = {
        'train_history': result['train_history'], 'val_history': result['val_history'],
        'best_iteration': result['best_iteration'],
    }
    metrics_list.append({
        'Model': run_name, 'Started At': result['started_at'],
        'Params': result['params'], 'Model File': model_file,
        'Cleanlab': cleanlab_enabled,
        'Seed': SEED,
        'Features': result['feature_columns'],
        'Best Iteration': result['best_iteration'],
        **(extra_metrics or {}),
        'Train AUC': result['train_auc'], 'Val AUC': result['val_auc'],
        'Fit Time (s)': round(result['fit_time'], 3),
    })

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors


class FullFeatureEngineering(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        n_splits=5,
        k_neighbors=15,
        n_pca_components=15,
        random_state=42,
    ):
        self.n_splits = n_splits
        self.k_neighbors = k_neighbors
        self.n_pca_components = n_pca_components
        self.random_state = random_state

    def _compute_unsupervised_aggs(self, X_df):
        vals = X_df.values
        total = vals.sum(axis=1)
        mean = vals.mean(axis=1)
        mean_safe = np.where(mean == 0, np.nan, mean)
        total_safe = np.where(total == 0, np.nan, total)

        aggs = {
            'total': total,
            'count_zeros': (vals == 0).sum(axis=1),
            'relative_std': vals.std(axis=1) / mean_safe,
            'max_share': vals.max(axis=1) / total_safe,
            'max': vals.max(axis=1),
            'log1p_sum': np.log1p(vals).sum(axis=1),
        }
        return pd.DataFrame(aggs, index=X_df.index).fillna(0)

    def fit(self, X, y):
        X_mat = np.asarray(X)
        y_mat = np.asarray(y)

        X_tr0, X_tr1 = X_mat[y_mat == 0], X_mat[y_mat == 1]

        # 1. Центроиды
        self.mu0_ = X_tr0.mean(axis=0)
        self.mu1_ = X_tr1.mean(axis=0)
        self.w_ = self.mu1_ - self.mu0_
        w_norm = np.linalg.norm(self.w_)
        self.w_norm_ = w_norm if w_norm > 0 else 1e-8

        # 2. Дискриминантные модели (LDA & QDA)
        self.lda_ = LinearDiscriminantAnalysis(
            solver='lsqr', shrinkage='auto'
        ).fit(X_mat, y_mat)
        self.qda_ = QuadraticDiscriminantAnalysis(reg_param=0.05).fit(
            X_mat, y_mat
        )

        # 3. Ковариации (LedoitWolf)
        self.cov0_ = LedoitWolf().fit(X_tr0)
        self.cov1_ = LedoitWolf().fit(X_tr1)

        # 4. Подпространства (PCA)
        n_comp0 = min(self.n_pca_components, X_tr0.shape[0] - 1)
        n_comp1 = min(self.n_pca_components, X_tr1.shape[0] - 1)
        self.pca0_ = PCA(
            n_components=n_comp0, random_state=self.random_state
        ).fit(X_tr0)
        self.pca1_ = PCA(
            n_components=n_comp1, random_state=self.random_state
        ).fit(X_tr1)

        # 5. Локальная топология (KNN)
        self.nn0_ = NearestNeighbors(
            n_neighbors=self.k_neighbors, n_jobs=-1
        ).fit(X_tr0)
        self.nn1_ = NearestNeighbors(
            n_neighbors=self.k_neighbors, n_jobs=-1
        ).fit(X_tr1)
        self.nn_all_ = NearestNeighbors(
            n_neighbors=self.k_neighbors + 1, n_jobs=-1
        ).fit(X_mat)
        self.y_mat_ = y_mat

        return self

    def fit_transform(self, X, y):
        X_df = pd.DataFrame(X) if isinstance(X, np.ndarray) else X.copy()
        X_mat = X_df.values
        y_mat = np.asarray(y)

        # Обучаем глобальные параметры на текущем датасете (Train или Train+Val)
        self.fit(X_mat, y_mat)

        # 1. Неразмеченные агрегаты
        df_aggs = self._compute_unsupervised_aggs(X_df)

        # 2. OOF Геометрия
        skf = StratifiedKFold(
            n_splits=self.n_splits, shuffle=True, random_state=self.random_state
        )

        cols_geom = [
            'd_euclid_diff',
            'proj_w',
            'cos_w',
            'lda_score',
            'qda_score',
            'd_mahal_0',
            'd_mahal_1',
            'd_mahal_diff',
            'pca_subspace_diff',
            'knn_dist_diff_k15',
            'knn_prob_class1_k15',
        ]
        oof_geom = np.zeros((len(X_mat), len(cols_geom)))

        for train_idx, val_idx in skf.split(X_mat, y_mat):
            X_tr, y_tr = X_mat[train_idx], y_mat[train_idx]
            X_va = X_mat[val_idx]
            X_tr0, X_tr1 = X_tr[y_tr == 0], X_tr[y_tr == 1]

            # Центроиды и Евклид
            mu0, mu1 = X_tr0.mean(axis=0), X_tr1.mean(axis=0)
            w = mu1 - mu0
            w_norm = np.linalg.norm(w) if np.linalg.norm(w) > 0 else 1e-8

            d0 = np.linalg.norm(X_va - mu0, axis=1)
            d1 = np.linalg.norm(X_va - mu1, axis=1)
            proj_w = X_va @ w
            x_norms = np.where(
                np.linalg.norm(X_va, axis=1) == 0,
                1e-8,
                np.linalg.norm(X_va, axis=1),
            )
            cos_w = proj_w / (x_norms * w_norm)

            # LDA & QDA
            lda = LinearDiscriminantAnalysis(
                solver='lsqr', shrinkage='auto'
            ).fit(X_tr, y_tr)
            qda = QuadraticDiscriminantAnalysis(reg_param=0.05).fit(X_tr, y_tr)
            lda_s = lda.decision_function(X_va)
            qda_s = qda.decision_function(X_va)

            # Mahalanobis
            cov0, cov1 = LedoitWolf().fit(X_tr0), LedoitWolf().fit(X_tr1)
            dm0 = np.sqrt(cov0.mahalanobis(X_va))
            dm1 = np.sqrt(cov1.mahalanobis(X_va))

            # PCA Subspaces
            n_comp0 = min(self.n_pca_components, X_tr0.shape[0] - 1)
            n_comp1 = min(self.n_pca_components, X_tr1.shape[0] - 1)
            pca0 = PCA(
                n_components=n_comp0, random_state=self.random_state
            ).fit(X_tr0)
            pca1 = PCA(
                n_components=n_comp1, random_state=self.random_state
            ).fit(X_tr1)

            rec0 = pca0.inverse_transform(pca0.transform(X_va))
            rec1 = pca1.inverse_transform(pca1.transform(X_va))
            pca_d0 = np.linalg.norm(X_va - rec0, axis=1)
            pca_d1 = np.linalg.norm(X_va - rec1, axis=1)

            # KNN Топология
            nn0 = NearestNeighbors(
                n_neighbors=self.k_neighbors, n_jobs=-1
            ).fit(X_tr0)
            nn1 = NearestNeighbors(
                n_neighbors=self.k_neighbors, n_jobs=-1
            ).fit(X_tr1)
            knn_d0 = nn0.kneighbors(X_va)[0].mean(axis=1)
            knn_d1 = nn1.kneighbors(X_va)[0].mean(axis=1)

            nn_all = NearestNeighbors(
                n_neighbors=self.k_neighbors + 1, n_jobs=-1
            ).fit(X_tr)
            indices = nn_all.kneighbors(X_va, return_distance=False)
            knn_prob1 = np.mean(y_tr[indices] == 1, axis=1)

            oof_geom[val_idx] = np.column_stack([
                d0 - d1,
                proj_w,
                cos_w,
                lda_s,
                qda_s,
                dm0,
                dm1,
                dm0 - dm1,
                pca_d0 - pca_d1,
                knn_d0 - knn_d1,
                knn_prob1,
            ])

        df_geom = pd.DataFrame(oof_geom, columns=cols_geom, index=X_df.index)
        return pd.concat([X_df, df_aggs, df_geom], axis=1)

    def transform(self, X):
        X_df = pd.DataFrame(X) if isinstance(X, np.ndarray) else X.copy()
        X_mat = X_df.values

        # 1. Неразмеченные агрегаты
        df_aggs = self._compute_unsupervised_aggs(X_df)

        # 2. Применение глобально обученных трансформеров
        d0 = np.linalg.norm(X_mat - self.mu0_, axis=1)
        d1 = np.linalg.norm(X_mat - self.mu1_, axis=1)
        proj_w = X_mat @ self.w_
        x_norms = np.where(
            np.linalg.norm(X_mat, axis=1) == 0,
            1e-8,
            np.linalg.norm(X_mat, axis=1),
        )
        cos_w = proj_w / (x_norms * self.w_norm_)

        lda_s = self.lda_.decision_function(X_mat)
        qda_s = self.qda_.decision_function(X_mat)

        dm0 = np.sqrt(self.cov0_.mahalanobis(X_mat))
        dm1 = np.sqrt(self.cov1_.mahalanobis(X_mat))

        rec0 = self.pca0_.inverse_transform(self.pca0_.transform(X_mat))
        rec1 = self.pca1_.inverse_transform(self.pca1_.transform(X_mat))
        pca_d0 = np.linalg.norm(X_mat - rec0, axis=1)
        pca_d1 = np.linalg.norm(X_mat - rec1, axis=1)

        knn_d0 = self.nn0_.kneighbors(X_mat)[0].mean(axis=1)
        knn_d1 = self.nn1_.kneighbors(X_mat)[0].mean(axis=1)

        indices = self.nn_all_.kneighbors(X_mat, return_distance=False)
        knn_prob1 = np.mean(self.y_mat_[indices] == 1, axis=1)

        cols_geom = [
            'd_euclid_diff',
            'proj_w',
            'cos_w',
            'lda_score',
            'qda_score',
            'd_mahal_0',
            'd_mahal_1',
            'd_mahal_diff',
            'pca_subspace_diff',
            'knn_dist_diff_k15',
            'knn_prob_class1_k15',
        ]

        res_geom = np.column_stack([
            d0 - d1,
            proj_w,
            cos_w,
            lda_s,
            qda_s,
            dm0,
            dm1,
            dm0 - dm1,
            pca_d0 - pca_d1,
            knn_d0 - knn_d1,
            knn_prob1,
        ])

        df_geom = pd.DataFrame(res_geom, columns=cols_geom, index=X_df.index)
        return pd.concat([X_df, df_aggs, df_geom], axis=1)

### SVC со взвешиванием признаков + перебор гипервараметров

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score
from datetime import datetime
import json
import time
import optuna
from sklearn.preprocessing import Normalizer

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# 1. КАСТОМНЫЙ ТРАНСФОРМЕР ВЗВЕШИВАНИЯ ПРИЗНАКОВ (FEATURE WEIGHTING)
# ==============================================================================

class UnivariateFeatureWeighter(BaseEstimator, TransformerMixin):
    """
    Трансформер взвешивает признаки перед RBF SVC на основе их одномерного ROC-AUC.
    w_j = |AUC_j - 0.5|^alpha
    
    При alpha = 0.0 -> все веса = 1.0 (обычный SVC)
    При alpha > 0.0 -> сильные признаки получают больший вес в RBF-расстоянии
    """
    def __init__(self, alpha=0.5, score_type='auc', top_k=None):
        self.alpha = alpha
        self.score_type = score_type
        self.top_k = top_k  # Можно совместить с top-K отбором (None = оставить все)
        self.weights_ = None
        self.selected_indices_ = None

    def fit(self, X, y):
        X_arr = np.asarray(X)
        y_arr = np.asarray(y)
        n_features = X_arr.shape[1]
        scores = np.zeros(n_features)

        if self.score_type == 'auc':
            for j in range(n_features):
                feature_col = X_arr[:, j]
                # Считаем ROC-AUC для отдельного признака
                try:
                    auc = roc_auc_score(y_arr, feature_col)
                except ValueError:
                    auc = 0.5
                # Сигнал — отклонение от 0.5 (случайного угадывания)
                scores[j] = abs(auc - 0.5)
                
        elif self.score_type == 'd_stat':
            # Статистика Коэна (d_j = |mean_0 - mean_1| / std_j)
            mask_0 = (y_arr == 0)
            mask_1 = (y_arr == 1)
            std_total = np.std(X_arr, axis=0) + 1e-8
            mean_diff = np.abs(np.mean(X_arr[mask_0], axis=0) - np.mean(X_arr[mask_1], axis=0))
            scores = mean_diff / std_total

        # Отбор Top-K если задан
        if self.top_k is not None and self.top_k < n_features:
            self.selected_indices_ = np.argsort(scores)[::-1][:self.top_k]
            scores = scores[self.selected_indices_]
        else:
            self.selected_indices_ = np.arange(n_features)

        # Вычисление весов w_j = score^alpha
        if self.alpha == 0.0:
            self.weights_ = np.ones_like(scores)
        else:
            # Нормируем максимальный вес в 1.0 для стабильности RBF gamma
            max_score = np.max(scores) + 1e-8
            norm_scores = scores / max_score
            self.weights_ = np.power(norm_scores, self.alpha)

        return self

    def transform(self, X):
        X_arr = np.asarray(X)
        # Фильтрация по top_k
        X_sub = X_arr[:, self.selected_indices_]
        # Взвешивание z_j <- z_j * w_j
        return X_sub * self.weights_

In [ ]:
# ==============================================================================
# 2. ПОДБОР HYPERPARAMETERS ЧЕРЕЗ OPTUNA (ALPHA + TOP_K + SVC)
# ==============================================================================


# Проверки данных перед стартом
for name, features in [('train', X_train), ('val', X_val), ('test', X_test)]:
    if not np.isfinite(features.to_numpy()).all() or (features < 0).any().any():
        raise ValueError(f'{name}: нужны конечные неотрицательные признаки')
    if list(features.columns) != list(X_train.columns):
        raise ValueError(f'{name}: порядок признаков отличается от train')

print('1/2. Расчет тяжелых геометрических и OOF признаков (один раз)...')

# log1p и сборка FullFeatureEngineering до Optuna
fe = FullFeatureEngineering(n_splits=5, random_state=SEED)

X_train_fe = fe.fit_transform(np.log1p(X_train), y_train)
X_val_fe = fe.transform(np.log1p(X_val))
X_test_fe = fe.transform(np.log1p(X_test))

print(
    f'Признаков до FE: {X_train.shape[1]} | Признаков после FE: {X_train_fe.shape[1]}'
)

# ==============================================================================
# 3. ПОДБОР HYPERPARAMETERS ЧЕРЕЗ OPTUNA
# ==============================================================================

print('2/2. Старт быстрого подбора Weighted RBF SVC...')

def objective(trial):
    # FullFeatureEngineering adds 17 columns in the current baseline.
    n_features_total = X_train.shape[1] + 17

    # --- Параметры взвешивания ---
    alpha = trial.suggest_float('alpha', 0.0, 1.5)  # 0.0 = без весов, 0.25, 0.5, 1.0, 1.5
    score_type = trial.suggest_categorical('score_type', ['auc', 'd_stat'])
    
    use_top_k = trial.suggest_categorical('use_top_k', [True, False])
    top_k = trial.suggest_int('top_k', 150, n_features_total) if use_top_k else None

    # --- Параметры RBF SVC ---
    C = trial.suggest_float('C', 1e-1, 100.0, log=True)
    gamma_type = trial.suggest_categorical('gamma_type', ['scale', 'auto', 'custom'])
    gamma = trial.suggest_float('gamma_value', 1e-4, 1.0, log=True) if gamma_type == 'custom' else gamma_type

    pipeline = Pipeline([
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        # fit_transform creates inner-OOF supervised columns for outer-train.
        ('fe', FullFeatureEngineering(n_splits=5, random_state=SEED)),
        ('scaler', StandardScaler()),
        ('weighter', UnivariateFeatureWeighter(alpha=alpha, score_type=score_type, top_k=top_k)),
        # ('l2_sphere', Normalizer(norm='l2')),
        ('svc', SVC(C=C, kernel='rbf', gamma=gamma, random_state=SEED))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=60, timeout=500, show_progress_bar=True)

print("\n=== ЛУЧШИЕ ПАРАМЕТРЫ WEIGHTED RBF SVC ===")
print(f"Best CV ROC-AUC: {study.best_value:.5f}")
for k, v in study.best_params.items():
    print(f" - {k}: {v}")


In [ ]:
# ==============================================================================
# 3. ФИНАЛЬНОЕ ОБУЧЕНИЕ И КАЛИБРОВКА ЛУЧШЕЙ МОДЕЛИ
# ==============================================================================

bp = study.best_params

best_weighter = UnivariateFeatureWeighter(
    alpha=bp['alpha'],
    score_type=bp['score_type'],
    top_k = bp.get('top_k', None) if bp.get('use_top_k', False) else None
)
gamma_best = bp.get('gamma_value', bp.get('gamma_type')) if bp.get('gamma_type') == 'custom' else bp.get('gamma_type')

best_svc_pipeline = Pipeline([
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('fe', FullFeatureEngineering(n_splits=5)),
    ('scaler', StandardScaler()),
    ('weighter', best_weighter),
    # ('l2_sphere', Normalizer(norm='l2')),
    ('svc', SVC(C=bp['C'], kernel='rbf', gamma=gamma_best, random_state=SEED))
])

svc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
svc_cv_scores = cross_val_score(best_svc_pipeline, X_train, y_train, cv=svc_cv, scoring='roc_auc', n_jobs=-1)

svc_started_at = datetime.now().astimezone()
svc_start = time.perf_counter()
svc_model = clone(best_svc_pipeline).fit(X_train, y_train)
svc_fit_time = time.perf_counter() - svc_start

svc_train_scores = svc_model.decision_function(X_train)
svc_val_scores = svc_model.decision_function(X_val)

svc_metrics = pd.DataFrame([{
    'Model': f"Weighted RBF SVC (alpha={bp['alpha']:.2f}, {bp['score_type']})",
    'CV AUC': svc_cv_scores.mean(),
    'CV STD': svc_cv_scores.std(),
    'Train AUC': roc_auc_score(y_train, svc_train_scores),
    'Val AUC': roc_auc_score(y_val, svc_val_scores),
    'Fit Time (s)': round(svc_fit_time, 3),
}])

display(svc_metrics)


### SVC без взвешивания

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import Normalizer

svc_pipeline = Pipeline([
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('fe', FullFeatureEngineering(n_splits=5)),
    ('scaler', StandardScaler()),
    ('svc', SVC(C=10, kernel='rbf', gamma='scale')),
])
svc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, features in [('train', X_train), ('val', X_val),
                       ('test', X_test)]:
    if not np.isfinite(features.to_numpy()).all() or (features < 0).any().any():
        raise ValueError(f'{name}: для этого эксперимента нужны конечные неотрицательные признаки')
    if list(features.columns) != list(X_train.columns):
        raise ValueError(f'{name}: состав или порядок признаков отличается от train')
print(f'SVC: {X_train.shape[1]} признаков')

SVC: 512 признаков


In [12]:
svc_cv_scores = cross_val_score(
    svc_pipeline, X_train, y_train,
    cv=svc_cv, scoring='roc_auc', n_jobs=1,
)
print('CV AUC по фолдам:', np.round(svc_cv_scores, 5))
print(f'CV AUC: {svc_cv_scores.mean():.5f} ± {svc_cv_scores.std():.5f}')

CV AUC по фолдам: [0.92536 0.93353 0.9281  0.93005 0.92289]
CV AUC: 0.92799 ± 0.00369


In [13]:
# Калибруется весь Pipeline: scaler обучается заново внутри каждого фолда.
# ensemble=False: итоговая SVC обучается на всём train.
svc_started_at = datetime.now().astimezone()
svc_start = time.perf_counter()
svc_model = clone(svc_pipeline).fit(X_train, y_train)
svc_fit_time = time.perf_counter() - svc_start

svc_train_scores = svc_model.decision_function(X_train)
svc_val_scores = svc_model.decision_function(X_val)

svc_metrics = pd.DataFrame([{
    'Model': 'log1p + StandardScaler + RBF SVC (extended)',
    'CV AUC (decision_function)': svc_cv_scores.mean(),
    'CV STD': svc_cv_scores.std(),
    'Train AUC': roc_auc_score(y_train, svc_train_scores),
    'Val AUC': roc_auc_score(y_val, svc_val_scores),
    'Fit Time (s)': round(svc_fit_time, 3),
}])

svc_run_dir = LOG_DIR / f'svc_{svc_started_at:%Y-%m-%d_%H-%M-%S_%f}'
# svc_model_path = save_model(svc_model, svc_run_dir / 'model.joblib')
# svc_log = {
#     'run_started_at': svc_started_at.isoformat(),
#     'feature_columns': list(X_train.columns),
#     'params': {'C': 10, 'kernel': 'rbf', 'gamma': 'scale'},
#     'preprocessing': ['log1p', 'StandardScaler'],
#     'calibration': {'method': 'sigmoid', 'ensemble': False, 'folds': 5},
#     'seed': SEED,
#     'cv_scores': svc_cv_scores.tolist(),
#     'results': [{
#         **json.loads(svc_metrics.to_json(orient='records'))[0],
#         'Model File': 'model.joblib',
#     }],
# }
# (svc_run_dir / 'results.json').write_text(
#     json.dumps(svc_log, ensure_ascii=False, indent=2), encoding='utf-8',
# )
# pd.DataFrame({
#     'row_id': val_df['row_id'], 'target': y_val, 'prediction': svc_val_probs,
# }).to_csv(svc_run_dir / 'validation_predictions.csv', index=False)
# print(f'Результаты: {svc_run_dir.resolve()}')
display(svc_metrics)


,Model,CV AUC (decision_function),CV STD,Train AUC,Val AUC,Fit + Calibration Time (s)
0,log1p + StandardScaler + RBF SVC (extended),0.927986,0.003686,1.0,0.950319,83.936


## Leakage-safe geometry, density likelihoods and block kernels

The cells below implement the ordered ablation requested for this notebook.  Every
supervised feature in an outer-fold training matrix is generated by an additional
inner OOF split; outer validation rows are transformed only by objects fitted on
the corresponding outer training part.  Expensive fold matrices are cached before
Optuna tunes SVC/kernel parameters.  Model selection uses `decision_function` only.


In [10]:
from svc_experiments import (
    PPCA_COMPONENTS,
    CoordinateLikelihoodFeatures,
    cache_outer_folds,
    experiment_row,
    feature_quality_table,
    fit_final_and_predict,
    fit_validation_blocks,
    optimize_svc,
    validation_auc,
)

# Set to False only for a quick inspection of the notebook structure.
RUN_LEAKAGE_SAFE_EXPERIMENTS = False  # superseded; prevents automatic submission
ABLATION_TRIALS = 15
BLOCK_KERNEL_TRIALS = 60
RUN_FACTOR_ANALYSIS = False  # Experiment 7 is deliberately optional/expensive.

# Predetermined (not validation-selected) robust WoE setting. Alternative settings
# can be compared by changing these values and rerunning the complete outer CV.
COORD_N_BINS = 8
COORD_ALPHA = 5.0


### Individual geometry quality (strict OOF train vs untouched validation)


In [11]:
# Individual OOF -> validation transfer for engineered geometry.
# In particular this prints Spearman correlations for all new Mahalanobis columns.
geometry_quality, fitted_quality_transformer = feature_quality_table(
    X_train, y_train, X_val, y_val, seed=SEED,
)
new_geometry_prefixes = (
    'mahal_', 'rda_', 'mid_', 'fisher_', 'ppca_', 'fa_'
)
display(geometry_quality[
    geometry_quality['feature'].str.startswith(new_geometry_prefixes)
].reset_index(drop=True))

transfer_warning = geometry_quality[
    geometry_quality['OOF -> validation gap'].abs() > 0.05
]
if len(transfer_warning):
    print('Features with unusually large OOF -> validation gap:')
    display(transfer_warning)


,feature,OOF Spearman,OOF single-feature ROC-AUC,validation single-feature ROC-AUC,OOF -> validation gap
0,ppca_llr_96,0.657693,0.879719,0.899617,0.019898
1,rda_norm_diff_0p0,0.655977,0.878728,0.884069,0.005341
2,fisher_cos,0.655621,0.878523,0.884184,0.005661
3,fisher_axis_snr,0.655621,0.878523,0.884184,0.005661
4,ppca_llr_64,0.653590,0.877350,0.896008,0.018658
5,ppca_llr_32,0.647427,0.873792,0.888240,0.014448
6,fisher_axis,0.644725,0.872232,0.876288,0.004056
7,rda_diff_0p0,0.644717,0.872227,0.876288,0.004061
8,mahal_sq_diff,0.641184,0.870188,0.888151,0.017963
9,rda_diff_1p0,0.641184,0.870188,0.888151,0.017963


### Coordinate LLR smoothing sensitivity (diagnostic only)

Four representative `(n_bins, alpha)` pairs cover the requested ranges without a
large Cartesian grid. They are reported OOF and on validation, but do not tune the
main experiment; changing the main setting requires rerunning the complete outer CV.


In [12]:
coord_sensitivity_rows = []
for n_bins, alpha in [(6, 1.0), (8, 2.0), (12, 5.0), (16, 10.0)]:
    coord = CoordinateLikelihoodFeatures(
        n_bins=n_bins, alpha=alpha, include_monotonic=True,
        include_groups=True, n_splits=5, random_state=SEED,
    )
    coord_oof = coord.fit_transform(np.log1p(X_train.to_numpy()), y_train)
    coord_val = coord.transform(np.log1p(X_val.to_numpy()))
    for feature in ['coord_llr_total', 'coord_llr_monotonic_total']:
        coord_sensitivity_rows.append({
            'n_bins': n_bins, 'alpha': alpha, 'feature': feature,
            'OOF_AUC': roc_auc_score(y_train, coord_oof[feature]),
            'validation_AUC': roc_auc_score(y_val, coord_val[feature]),
        })
coord_sensitivity = pd.DataFrame(coord_sensitivity_rows)
display(coord_sensitivity)


,n_bins,alpha,feature,OOF_AUC,validation_AUC
0,6,1.0,coord_llr_total,0.845393,0.840408
1,6,1.0,coord_llr_monotonic_total,0.845149,0.840102
2,8,2.0,coord_llr_total,0.846682,0.840408
3,8,2.0,coord_llr_monotonic_total,0.846387,0.840064
4,12,5.0,coord_llr_total,0.846720,0.842474
5,12,5.0,coord_llr_monotonic_total,0.846201,0.842079
6,16,10.0,coord_llr_total,0.846420,0.843686
7,16,10.0,coord_llr_monotonic_total,0.846439,0.842207


### Experiments 0–4: ordered ordinary-RBF ablation


In [13]:
experiment_rows = []
experiment_artifacts = {}

if RUN_LEAKAGE_SAFE_EXPERIMENTS:
    # Ordered ablation. Validation is reported only after CV selected the SVC params.
    ablations = [
        ('Baseline', 'baseline'),
        ('Experiment 1: enhanced geometry', 'enhanced'),
        ('Experiment 2: + PPCA LLR', 'ppca'),
        ('Experiment 3: + coordinate LLR', 'coord'),
        ('Experiment 4: + monotonic/group LLR', 'coord_monotonic'),
    ]

    for experiment_name, config in ablations:
        started = time.perf_counter()
        folds = cache_outer_folds(
            X_train, y_train, config, seed=SEED,
            n_bins=COORD_N_BINS, alpha=COORD_ALPHA,
        )
        study, params, cv_scores = optimize_svc(
            folds, kernel_type='rbf', n_trials=ABLATION_TRIALS, seed=SEED,
        )
        builder, blocks, geom_names, density_names = fit_validation_blocks(
            X_train, y_train, X_val, config, seed=SEED,
            n_bins=COORD_N_BINS, alpha=COORD_ALPHA,
        )
        val_auc, val_decision = validation_auc(blocks, y_train, y_val, params)
        elapsed = time.perf_counter() - started
        experiment_rows.append(experiment_row(
            experiment_name, cv_scores, val_auc, X_train.shape[1],
            len(geom_names) + len(density_names), params, elapsed,
        ))
        experiment_artifacts[experiment_name] = {
            'config': config, 'folds': folds, 'params': params,
            'cv_scores': cv_scores, 'blocks': blocks,
        }
        display(pd.DataFrame(experiment_rows).tail(1))

    ablation_results = pd.DataFrame(experiment_rows)
    display(ablation_results.sort_values('CV_mean', ascending=False).reset_index(drop=True))


  cache baseline: outer fold 1/5
  cache baseline: outer fold 2/5
  cache baseline: outer fold 3/5
  cache baseline: outer fold 4/5
  cache baseline: outer fold 5/5


c:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\venvmy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 6. Best value: 0.930067: 100%|██████████| 15/15 [00:55<00:00,  3.72s/it]


,experiment,CV_mean,CV_std,validation_auc,n_features_raw,n_features_geometry,C,kernel_type,gamma_raw,gamma_geom,beta,fit_time
0,Baseline,0.930067,0.004793,0.95,512,17,3.447606,rbf,0.00311,NaN,NaN,229.062446


  cache enhanced: outer fold 1/5
  cache enhanced: outer fold 2/5
  cache enhanced: outer fold 3/5
  cache enhanced: outer fold 4/5
  cache enhanced: outer fold 5/5


Best trial: 6. Best value: 0.930864: 100%|██████████| 15/15 [00:58<00:00,  3.90s/it]


...

,experiment,CV_mean,CV_std,validation_auc,n_features_raw,n_features_geometry,C,kernel_type,gamma_raw,gamma_geom,beta,fit_time
3,Experiment 3: + coordinate LLR,0.930839,0.004951,0.950969,512,43,3.447606,rbf,0.00311,NaN,NaN,321.508806


  cache coord_monotonic: outer fold 1/5
  cache coord_monotonic: outer fold 2/5
  cache coord_monotonic: outer fold 3/5
  cache coord_monotonic: outer fold 4/5
  cache coord_monotonic: outer fold 5/5


Best trial: 6. Best value: 0.930905: 100%|██████████| 15/15 [01:28<00:00,  5.91s/it]


,experiment,CV_mean,CV_std,validation_auc,n_features_raw,n_features_geometry,C,kernel_type,gamma_raw,gamma_geom,beta,fit_time
4,Experiment 4: + monotonic/group LLR,0.930905,0.004853,0.950918,512,52,3.447606,rbf,0.00311,NaN,NaN,336.226495


,experiment,CV_mean,CV_std,validation_auc,n_features_raw,n_features_geometry,C,kernel_type,gamma_raw,gamma_geom,beta,fit_time
0,Experiment 2: + PPCA LLR,0.930927,0.005010,0.951569,512,54,3.447606,rbf,0.00311,NaN,NaN,296.674884
1,Experiment 4: + monotonic/group LLR,0.930905,0.004853,0.950918,512,52,3.447606,rbf,0.00311,NaN,NaN,336.226495
2,Experiment 1: enhanced geometry,0.930864,0.004941,0.950982,512,42,3.447606,rbf,0.00311,NaN,NaN,233.980826
3,Experiment 3: + coordinate LLR,0.930839,0.004951,0.950969,512,43,3.447606,rbf,0.00311,NaN,NaN,321.508806
4,Baseline,0.930067,0.004793,0.950000,512,17,3.447606,rbf,0.00311,NaN,NaN,229.062446


### Experiments 5–6: cached product and additive block kernels


In [14]:
if RUN_LEAKAGE_SAFE_EXPERIMENTS:
    # Pick the feature family by outer-CV only. A 0.0002 fluctuation is not treated
    # as meaningful: among configurations within 0.0005 of the best mean, prefer
    # the one with the smaller CV standard deviation.
    ordinary = pd.DataFrame(experiment_rows)
    near_best = ordinary[ordinary.CV_mean >= ordinary.CV_mean.max() - 5e-4]
    selected_name = near_best.sort_values(['CV_std', 'CV_mean'], ascending=[True, False]).iloc[0].experiment
    selected = experiment_artifacts[selected_name]
    print('Feature family selected from CV:', selected_name)

    for kernel_type, label in [
        ('product', 'Experiment 5: product/block RBF'),
        ('additive', 'Experiment 6: additive multi-kernel RBF'),
    ]:
        started = time.perf_counter()
        study, params, cv_scores = optimize_svc(
            selected['folds'], kernel_type=kernel_type,
            n_trials=BLOCK_KERNEL_TRIALS, seed=SEED,
        )
        val_auc, val_decision = validation_auc(
            selected['blocks'], y_train, y_val, params,
        )
        elapsed = time.perf_counter() - started
        n_geom = selected['blocks'][2].shape[1] + selected['blocks'][4].shape[1]
        experiment_rows.append(experiment_row(
            label, cv_scores, val_auc, X_train.shape[1], n_geom, params, elapsed,
        ))
        experiment_artifacts[label] = {
            **selected, 'params': params, 'cv_scores': cv_scores,
        }

    experiment_results = pd.DataFrame(experiment_rows)
    display(experiment_results.sort_values('CV_mean', ascending=False).reset_index(drop=True))


Feature family selected from CV: Experiment 4: + monotonic/group LLR


Best trial: 56. Best value: 0.932587: 100%|██████████| 60/60 [14:13<00:00, 14.23s/it]


,experiment,CV_mean,CV_std,validation_auc,n_features_raw,n_features_geometry,C,kernel_type,gamma_raw,gamma_geom,beta,fit_time
0,Experiment 5: product/block RBF,0.932628,0.005023,0.953571,512,52,18.152351,product,0.002496,0.010154,NaN,806.193970
1,Experiment 6: additive multi-kernel RBF,0.932587,0.004439,0.952819,512,52,5.955451,additive,0.002935,0.003011,0.776777,867.918253
2,Experiment 2: + PPCA LLR,0.930927,0.005010,0.951569,512,54,3.447606,rbf,0.003110,NaN,NaN,296.674884
3,Experiment 4: + monotonic/group LLR,0.930905,0.004853,0.950918,512,52,3.447606,rbf,0.003110,NaN,NaN,336.226495
4,Experiment 1: enhanced geometry,0.930864,0.004941,0.950982,512,42,3.447606,rbf,0.003110,NaN,NaN,233.980826
5,Experiment 3: + coordinate LLR,0.930839,0.004951,0.950969,512,43,3.447606,rbf,0.003110,NaN,NaN,321.508806
6,Baseline,0.930067,0.004793,0.950000,512,17,3.447606,rbf,0.003110,NaN,NaN,229.062446


### Experiment 7 (optional): class-conditional Factor Analysis LLR


In [15]:
if RUN_LEAKAGE_SAFE_EXPERIMENTS and RUN_FACTOR_ANALYSIS:
    started = time.perf_counter()
    fa_folds = cache_outer_folds(X_train, y_train, 'fa', seed=SEED)
    fa_study, fa_params, fa_cv = optimize_svc(
        fa_folds, kernel_type='rbf', n_trials=ABLATION_TRIALS, seed=SEED,
    )
    _, fa_blocks, fa_geom_names, fa_density_names = fit_validation_blocks(
        X_train, y_train, X_val, 'fa', seed=SEED,
    )
    fa_val_auc, _ = validation_auc(fa_blocks, y_train, y_val, fa_params)
    experiment_rows.append(experiment_row(
        'Experiment 7: + FactorAnalysis LLR', fa_cv, fa_val_auc,
        X_train.shape[1], len(fa_geom_names) + len(fa_density_names),
        fa_params, time.perf_counter() - started,
    ))
    experiment_artifacts['Experiment 7: + FactorAnalysis LLR'] = {
        'config': 'fa', 'folds': fa_folds, 'params': fa_params,
        'cv_scores': fa_cv, 'blocks': fa_blocks,
    }
    experiment_results = pd.DataFrame(experiment_rows)
    display(experiment_results.sort_values('CV_mean', ascending=False))


### Final train + validation refit and Kaggle submission


In [ ]:
if RUN_LEAKAGE_SAFE_EXPERIMENTS:
    # Model was selected beforehand using leakage-safe CV
    # with validation used only as an independent sanity check.
    winner_name = 'Experiment 5: product/block RBF'
    winner_artifact = experiment_artifacts[winner_name]

    winner = experiment_results.loc[
        experiment_results['experiment'] == winner_name
    ].iloc[0]

    print('Final model:', winner_name)
    print(f'CV mean:        {winner.CV_mean:.6f}')
    print(f'CV std:         {winner.CV_std:.6f}')
    print(f'Validation AUC: {winner.validation_auc:.6f}')
    print('Parameters:', winner_artifact['params'])

    # Fit ONLY on train.csv.
    # validation.csv is not used for training.
    test_prediction, final_feature_builder, final_svc = fit_final_and_predict(
        X_train,
        y_train,
        X_test,
        config=winner_artifact['config'],
        params=winner_artifact['params'],
        seed=SEED,
        n_bins=COORD_N_BINS,
        alpha=COORD_ALPHA,
    )

    submission = pd.DataFrame({
        'row_id': test_df['row_id'].to_numpy(),
        'target': test_prediction,
    })

    submission_path = OUTPUT_DIR / 'submission_product_block_rbf.csv'
    submission.to_csv(submission_path, index=False)

    print('Saved:', submission_path.resolve())
    display(submission.head())

## Stage 2 — structural representations under repeated CV

This stage deliberately avoids supervised centroid geometry. It compares raw,
order-statistic, distribution-shape and robust-sorted spaces on the same 25 folds.
External validation is evaluation-only. No cell in this stage trains on validation,
uses it for early stopping, predicts test, or writes a submission.

Run Phase A first and inspect its summary before enabling Phases B–D.


In [23]:
from structural_experiments import (
    GaussianCopulaFeatures,
    build_repeated_distance_cache,
    evaluate_kernel,
    external_validation_kernel,
    fold_summary,
    metric_external_validation,
    optimize_structural_kernel,
    paired_diagnostics,
    permutation_invariance_diagnostic,
    prediction_complementarity,
    repeated_copula_cv,
    repeated_metric_cv,
    score_current_best_repeated,
)
from svc_experiments import fit_validation_blocks, validation_auc
from scipy.stats import pearsonr, spearmanr

N_SPLITS = 5
STRUCTURAL_SEEDS = [SEED + i for i in range(5)]

RUN_STRUCTURAL_PHASE_A = True
RUN_CURRENT_BEST_BASELINE = True  # configured below; expensive repeated nested OOF run
RUN_MONOTONE_PHASE_B = True       # enable only after inspecting Phase A
RUN_STRUCTURE_PHASE_C = True      # the cell still obeys the sorted-signal gate
FORCE_STRUCTURE_PHASE_C = False   # True runs block sorting despite weak standalone Sorted
RUN_COPULA_PHASE_D = True         # last/optional branch

SINGLE_SPACE_TRIALS = 40
TWO_BLOCK_TRIALS = 60
THREE_BLOCK_TRIALS = 70

# Previous-stage winner recovered from the saved notebook results:
# Experiment 4 feature family + Experiment 5 product/block RBF.
# Values are the precision printed in the saved result table.
CURRENT_BEST_CONFIG = 'coord_monotonic'
CURRENT_BEST_PARAMS = {
    'kernel_type': 'product',
    'C': 18.152351,
    'gamma_raw': 0.002496,
    'gamma_geom': 0.010154,
}

# Metric search controls. Allowed ranges are lr=1e-4..3e-3 and
# weight_decay=1e-6..1e-2; keep one setting per repeated-CV run to control cost.
METRIC_LR = 1e-3
METRIC_WEIGHT_DECAY = 1e-4
METRIC_MAX_EPOCHS = 400
METRIC_PATIENCE = 40


### Phase A — representation discovery (STOP after this phase)


In [12]:
structural_records = {}
structural_oof = {}

if RUN_STRUCTURAL_PHASE_A:
    representation_names = ('raw', 'sorted', 'shape', 'robust_sorted')
    structural_cache = build_repeated_distance_cache(
        X_train, y_train, STRUCTURAL_SEEDS, N_SPLITS,
        representations=representation_names,
    )

    phase_a_specs = [
        ('A. Raw only', ('raw',), 'single', SINGLE_SPACE_TRIALS),
        ('B. Sorted only', ('sorted',), 'single', SINGLE_SPACE_TRIALS),
        ('C. Shape only', ('shape',), 'single', SINGLE_SPACE_TRIALS),
        ('D. Robust Sorted only', ('robust_sorted',), 'single', SINGLE_SPACE_TRIALS),
        ('E. Raw + Sorted additive', ('raw', 'sorted'), 'additive', TWO_BLOCK_TRIALS),
        ('F. Raw + Sorted product', ('raw', 'sorted'), 'product', TWO_BLOCK_TRIALS),
        ('G. Raw + Robust Sorted additive', ('raw', 'robust_sorted'), 'additive', TWO_BLOCK_TRIALS),
        ('H. Raw + Sorted + Shape additive', ('raw', 'sorted', 'shape'), 'additive', THREE_BLOCK_TRIALS),
    ]

    for name, spaces, kernel, trials in phase_a_specs:
        started = time.perf_counter()
        study, params, aucs, oof = optimize_structural_kernel(
            structural_cache, spaces, kernel=kernel,
            n_trials=trials, seed=SEED,
        )
        val_auc, val_score = external_validation_kernel(
            X_train, y_train, X_val, y_val, params,
        )
        structural_records[name] = {
            'representation': ' + '.join(spaces), 'kernel': kernel,
            'params': params, 'auc_per_fold': aucs, 'oof': oof,
            'validation_auc': val_auc, 'validation_score': val_score,
            'fit_time': time.perf_counter() - started,
        }
        structural_oof[name] = oof
        print(name, f'CV={aucs.mean():.5f}', f'validation={val_auc:.5f}')


distance cache repeat 1/5, fold 1/5
distance cache repeat 1/5, fold 2/5
distance cache repeat 1/5, fold 3/5
distance cache repeat 1/5, fold 4/5
distance cache repeat 1/5, fold 5/5
distance cache repeat 2/5, fold 1/5
distance cache repeat 2/5, fold 2/5
distance cache repeat 2/5, fold 3/5
distance cache repeat 2/5, fold 4/5
distance cache repeat 2/5, fold 5/5
distance cache repeat 3/5, fold 1/5
distance cache repeat 3/5, fold 2/5
distance cache repeat 3/5, fold 3/5
distance cache repeat 3/5, fold 4/5
distance cache repeat 3/5, fold 5/5
distance cache repeat 4/5, fold 1/5
distance cache repeat 4/5, fold 2/5
distance cache repeat 4/5, fold 3/5
distance cache repeat 4/5, fold 4/5
distance cache repeat 4/5, fold 5/5
distance cache repeat 5/5, fold 1/5
distance cache repeat 5/5, fold 2/5
distance cache repeat 5/5, fold 3/5
distance cache repeat 5/5, fold 4/5
distance cache repeat 5/5, fold 5/5


c:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\venvmy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 22. Best value: 0.929299: 100%|██████████| 40/40 [02:17<00:00,  3.43s/it]


A. Raw only CV=0.92930 validation=0.94741


Best trial: 33. Best value: 0.835456: 100%|██████████| 40/40 [01:55<00:00,  2.88s/it]


B. Sorted only CV=0.83546 validation=0.84042


...

F. Raw + Sorted product CV=0.92973 validation=0.94821


Best trial: 42. Best value: 0.930695: 100%|██████████| 60/60 [04:27<00:00,  4.46s/it]


G. Raw + Robust Sorted additive CV=0.93070 validation=0.94448


Best trial: 66. Best value: 0.930334: 100%|██████████| 70/70 [08:12<00:00,  7.04s/it]


H. Raw + Sorted + Shape additive CV=0.93033 validation=0.94562


### Optional repeated-CV current-best baseline on identical folds


In [24]:
current_best_aucs = None
current_best_oof = None
current_best_validation_auc = np.nan
current_best_validation_score = None

if RUN_STRUCTURAL_PHASE_A and RUN_CURRENT_BEST_BASELINE:
    current_best_aucs, current_best_oof = score_current_best_repeated(
        X_train, y_train, STRUCTURAL_SEEDS,
        CURRENT_BEST_CONFIG, CURRENT_BEST_PARAMS, N_SPLITS,
    )
    _, current_blocks, _, _ = fit_validation_blocks(
        X_train, y_train, X_val, CURRENT_BEST_CONFIG, seed=SEED,
    )
    current_best_validation_auc, current_best_validation_score = validation_auc(
        current_blocks, y_train, y_val, CURRENT_BEST_PARAMS,
    )
    structural_records['Baseline Current Best'] = {
        'representation': CURRENT_BEST_CONFIG,
        'kernel': CURRENT_BEST_PARAMS['kernel_type'],
        'params': CURRENT_BEST_PARAMS,
        'auc_per_fold': current_best_aucs,
        'oof': current_best_oof,
        'validation_auc': current_best_validation_auc,
        'validation_score': current_best_validation_score,
        'fit_time': np.nan,
    }
    structural_oof['Baseline Current Best'] = current_best_oof


current best repeat=1, fold=1
current best repeat=1, fold=2
current best repeat=1, fold=3
current best repeat=1, fold=4
current best repeat=1, fold=5
current best repeat=2, fold=1
current best repeat=2, fold=2
current best repeat=2, fold=3
current best repeat=2, fold=4
current best repeat=2, fold=5
current best repeat=3, fold=1
current best repeat=3, fold=2
current best repeat=3, fold=3
current best repeat=3, fold=4
current best repeat=3, fold=5
current best repeat=4, fold=1
current best repeat=4, fold=2
current best repeat=4, fold=3
current best repeat=4, fold=4
current best repeat=4, fold=5
current best repeat=5, fold=1
current best repeat=5, fold=2
current best repeat=5, fold=3
current best repeat=5, fold=4
current best repeat=5, fold=5


### Phase A tables, paired tests and complementarity


In [25]:
if RUN_STRUCTURAL_PHASE_A:
    raw_name = 'A. Raw only'
    raw_aucs = structural_records[raw_name]['auc_per_fold']
    raw_oof = structural_records[raw_name]['oof']
    raw_validation_auc = structural_records[raw_name]['validation_auc']

    structural_rows = []
    for name, record in structural_records.items():
        structural_rows.append(fold_summary(
            name, record['representation'], record['kernel'],
            record['auc_per_fold'], record['validation_auc'], record['params'],
            baseline_raw=raw_aucs, baseline_current=current_best_aucs,
            fit_time=record['fit_time'],
        ))
    structural_summary = pd.DataFrame(structural_rows)

    # Fold-wise AUC and paired deltas are retained, not only aggregate statistics.
    foldwise_auc = pd.DataFrame({
        name: record['auc_per_fold'] for name, record in structural_records.items()
    })
    foldwise_auc.insert(0, 'fold_id', [
        f'seed_{repeat}_fold_{fold}'
        for repeat in range(len(STRUCTURAL_SEEDS))
        for fold in range(N_SPLITS)
    ])
    paired_vs_raw = pd.DataFrame([
        {'experiment': name, **paired_diagnostics(record['auc_per_fold'], raw_aucs)}
        for name, record in structural_records.items()
    ])

    corr_inputs = {
        'raw': structural_records['A. Raw only']['oof'],
        'sorted': structural_records['B. Sorted only']['oof'],
        'shape': structural_records['C. Shape only']['oof'],
    }
    if current_best_oof is not None:
        corr_inputs['current_best'] = current_best_oof
    prediction_correlations = prediction_complementarity(corr_inputs, y_train, reference='raw')

    corr_raw = {
        name: spearmanr(raw_oof.ravel(), record['oof'].ravel()).statistic
        for name, record in structural_records.items()
    }
    corr_current = {
        name: (np.nan if current_best_oof is None else
               spearmanr(current_best_oof.ravel(), record['oof'].ravel()).statistic)
        for name, record in structural_records.items()
    }
    structural_summary['prediction_corr_with_raw'] = structural_summary.experiment.map(corr_raw)
    structural_summary['prediction_corr_with_current'] = structural_summary.experiment.map(corr_current)
    structural_summary['auc_per_fold'] = structural_summary.experiment.map(
        {name: record['auc_per_fold'].tolist() for name, record in structural_records.items()}
    )

    display(structural_summary.sort_values('CV_mean', ascending=False).reset_index(drop=True))
    display(foldwise_auc)
    display(paired_vs_raw)
    display(prediction_correlations)

    # Stable selection rule. Validation is only a sanity check, never a tuner.
    candidates = structural_summary[
        (structural_summary.delta_vs_raw >= 0.002) &
        (structural_summary.wins_vs_raw >= 17) &
        (structural_summary.validation_auc >= raw_validation_auc)
    ]
    if len(candidates):
        recommended_structural = candidates.sort_values(
            ['CV_mean', 'CV_std'], ascending=[False, True]
        ).iloc[0]
        print('Recommended candidate based on repeated CV:', recommended_structural.experiment)
    else:
        recommended_structural = structural_summary.sort_values('CV_mean', ascending=False).iloc[0]
        print('No candidate passed the +0.002 / 17-of-25 stability rule.')
        print('Best diagnostic candidate:', recommended_structural.experiment)


,experiment,representation,kernel,CV_mean,CV_std,CV_median,CV_min,CV_max,validation_auc,params,fit_time,delta_vs_raw,wins_vs_raw,delta_vs_current_best,wins_vs_current_best,prediction_corr_with_raw,prediction_corr_with_current,auc_per_fold
0,Baseline Current Best,coord_monotonic,product,0.931039,0.007146,0.929910,0.916399,0.946728,0.953571,"{'kernel_type': 'product', 'C': 18.152351, 'ga...",NaN,0.001740,18,0.000000,0,0.986188,1.000000,"[0.9250755857898714, 0.9386652809271856, 0.932..."
1,G. Raw + Robust Sorted additive,raw + robust_sorted,additive,0.930695,0.007461,0.929563,0.913281,0.947358,0.944477,"{'kernel': 'additive', 'spaces': ('raw', 'robu...",277.853692,0.001396,18,-0.000343,10,0.992767,0.981847,"[0.9260991433610481, 0.9345238095238095, 0.931..."
2,E. Raw + Sorted additive,raw + sorted,additive,0.930569,0.007278,0.929201,0.913612,0.947043,0.945918,"{'kernel': 'additive', 'spaces': ('raw', 'sort...",237.301576,0.001270,18,-0.000469,11,0.995440,0.984031,"[0.9259259259259259, 0.9346970269589318, 0.931..."
3,H. Raw + Sorted + Shape additive,raw + sorted + shape,additive,0.930334,0.007423,0.929154,0.913439,0.947940,0.945625,"{'kernel': 'additive', 'spaces': ('raw', 'sort...",510.384737,0.001035,18,-0.000705,9,0.994301,0.983818,"[0.9262881078357269, 0.9351379440665155, 0.931..."
4,F. Raw + Sorted product,raw + sorted,product,0.929730,0.007459,0.928650,0.912935,0.946964,0.948214,"{'kernel': 'product', 'spaces': ('raw', 'sorte...",138.072106,0.000431,18,-0.001309,7,0.998470,0.988046,"[0.9237370874275636, 0.9355473670949862, 0.930..."
5,A. Raw only,raw,single,0.929299,0.007345,0.928886,0.913848,0.946224,0.947411,"{'kernel': 'single', 'spaces': ('raw',), 'C': ...",150.150013,0.000000,0,-0.001740,7,1.000000,0.986188,"[0.9235166288737717, 0.9346970269589316, 0.929..."
6,B. Sorted only,sorted,single,0.835456,0.016252,0.835743,0.795351,0.862150,0.840421,"{'kernel': 'single', 'spaces': ('sorted',), 'C...",126.404065,-0.093843,0,-0.095583,0,0.796523,0.768375,"[0.7953514739229025, 0.8480410682791635, 0.826..."
7,C. Shape only,shape,single,0.835187,0.014565,0.836467,0.799367,0.856182,0.838699,"{'kernel': 'single', 'spaces': ('shape',), 'C'...",122.068758,-0.094112,0,-0.095852,0,0.799901,0.770827,"[0.7993669690098262, 0.8492220962459058, 0.836..."
8,D. Robust Sorted only,robust_sorted,single,0.833542,0.016198,0.835176,0.795147,0.863190,0.826467,"{'kernel': 'single', 'spaces': ('robust_sorted...",119.218922,-0.095756,0,-0.097496,0,0.786119,0.754673,"[0.7951467624086671, 0.8364355001259763, 0.833..."


,fold_id,A. Raw only,B. Sorted only,C. Shape only,D. Robust Sorted only,E. Raw + Sorted additive,F. Raw + Sorted product,G. Raw + Robust Sorted additive,H. Raw + Sorted + Shape additive,Baseline Current Best
0,seed_0_fold_0,0.923517,0.795351,0.799367,0.795147,0.925926,0.923737,0.926099,0.926288,0.925076
1,seed_0_fold_1,0.934697,0.848041,0.849222,0.836436,0.934697,0.935547,0.934524,0.935138,0.938665
2,seed_0_fold_2,0.929123,0.826814,0.836971,0.833617,0.931516,0.930052,0.931485,0.931453,0.932335
3,seed_0_fold_3,0.931453,0.853112,0.845238,0.858418,0.934949,0.933469,0.935421,0.933894,0.936177
4,seed_0_fold_4,0.925768,0.844939,0.841569,0.837191,0.924745,0.925469,0.924052,0.924556,0.926320
5,seed_1_fold_0,0.944885,0.845333,0.844183,0.851773,0.945421,0.944901,0.945814,0.944035,0.945074
6,seed_1_fold_1,0.935689,0.862150,0.854812,0.863190,0.938161,0.936823,0.938650,0.937957,0.935784
7,seed_1_fold_2,0.929327,0.820673,0.817523,0.813287,0.931327,0.930682,0.930965,0.931548,0.932776
8,seed_1_fold_3,0.926446,0.820232,0.822121,0.814122,0.927989,0.926886,0.928162,0.927564,0.925202
9,seed_1_fold_4,0.913848,0.832861,0.836467,0.824751,0.913612,0.912935,0.913281,0.913439,0.916399


,experiment,mean_delta,std_delta,median_delta,wins,losses,ties,wilcoxon_p,delta_per_fold
0,A. Raw only,0.000000,0.000000,0.000000,0,0,25,1.000000e+00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,B. Sorted only,-0.093843,0.013110,-0.094955,0,25,0,5.960464e-08,"[-0.12816515495086922, -0.08665595867976816, -..."
2,C. Shape only,-0.094112,0.012085,-0.092734,0,25,0,5.960464e-08,"[-0.12414965986394555, -0.08547493071302581, -..."
3,D. Robust Sorted only,-0.095756,0.013084,-0.095506,0,25,0,5.960464e-08,"[-0.12836986646510462, -0.0982615268329553, -0..."
4,E. Raw + Sorted additive,0.001270,0.001490,0.001244,18,5,2,1.027304e-03,"[0.0024092970521542023, 1.1102230246251565e-16..."
5,F. Raw + Sorted product,0.000431,0.001050,0.000598,18,7,0,8.028611e-02,"[0.00022045855379182466, 0.0008503401360545615..."
6,G. Raw + Robust Sorted additive,0.001396,0.001754,0.001543,18,7,0,8.081198e-04,"[0.0025825144872764216, -0.0001732174351221083..."
7,H. Raw + Sorted + Shape additive,0.001035,0.001666,0.001150,18,7,0,8.364365e-03,"[0.002771478961955176, 0.00044091710758387137,..."
8,Baseline Current Best,0.001740,0.002176,0.001559,18,7,0,1.625359e-03,"[0.0015589569160996408, 0.003968253968253954, ..."


,score,pearson_with_raw,spearman_with_raw,top_quartile_disagreement_auc,mean_abs_rank_disagreement
0,raw,1.000000,1.000000,0.928759,0.000000
1,sorted,0.829313,0.796523,0.408675,0.137276
2,shape,0.810466,0.799901,0.402769,0.137178
3,current_best,0.974067,0.986188,0.936777,0.036773


No candidate passed the +0.002 / 17-of-25 stability rule.
Best diagnostic candidate: Baseline Current Best


### Phase B — monotone diagonal and low-rank ranking models


In [26]:
metric_records = {}
if RUN_MONOTONE_PHASE_B:
    metric_cv = repeated_metric_cv(
        X_train, y_train, STRUCTURAL_SEEDS,
        ranks=(0, 4, 8, 16, 32), n_splits=N_SPLITS,
        lr=METRIC_LR, weight_decay=METRIC_WEIGHT_DECAY,
        max_epochs=METRIC_MAX_EPOCHS, patience=METRIC_PATIENCE,
        batch_size=256, hybrid_bce=0.0,
    )
    metric_rows = []
    for rank, result in metric_cv.items():
        val_auc, val_score, refit_epochs = metric_external_validation(
            X_train, y_train, X_val, y_val, rank=rank, seed=SEED,
            lr=METRIC_LR, weight_decay=METRIC_WEIGHT_DECAY,
            max_epochs=METRIC_MAX_EPOCHS, patience=METRIC_PATIENCE,
            batch_size=256, hybrid_bce=0.0,
        )
        name = 'Diagonal monotone metric' if rank == 0 else f'Low-rank monotone metric r={rank}'
        diag = paired_diagnostics(result['auc_per_fold'], raw_aucs)
        corr_raw = spearmanr(result['oof'].ravel(), raw_oof.ravel()).statistic
        corr_current = np.nan if current_best_oof is None else spearmanr(
            result['oof'].ravel(), current_best_oof.ravel()
        ).statistic
        metric_rows.append({
            'experiment': name, 'rank': rank,
            'CV_mean': result['auc_per_fold'].mean(),
            'CV_std': result['auc_per_fold'].std(),
            'CV_median': np.median(result['auc_per_fold']),
            'CV_min': result['auc_per_fold'].min(),
            'CV_max': result['auc_per_fold'].max(),
            'delta_vs_raw': diag['mean_delta'], 'wins_vs_raw': diag['wins'],
            'losses_vs_raw': diag['losses'], 'ties_vs_raw': diag['ties'],
            'wilcoxon_p': diag['wilcoxon_p'], 'validation_auc': val_auc,
            'prediction_corr_with_raw': corr_raw,
            'prediction_corr_with_current': corr_current,
            'auc_per_fold': result['auc_per_fold'].tolist(),
            'mean_early_stop_epochs': result['mean_epochs'],
            'external_refit_epochs': refit_epochs,
        })
        metric_records[name] = {**result, 'validation_auc': val_auc, 'validation_score': val_score}
    metric_summary = pd.DataFrame(metric_rows).sort_values('CV_mean', ascending=False)
    display(metric_summary.reset_index(drop=True))


metric rank=0, repeat=1, fold=1
metric rank=0, repeat=1, fold=2
metric rank=0, repeat=1, fold=3
metric rank=0, repeat=1, fold=4
metric rank=0, repeat=1, fold=5
metric rank=0, repeat=2, fold=1
metric rank=0, repeat=2, fold=2
metric rank=0, repeat=2, fold=3
metric rank=0, repeat=2, fold=4
metric rank=0, repeat=2, fold=5
metric rank=0, repeat=3, fold=1
metric rank=0, repeat=3, fold=2
metric rank=0, repeat=3, fold=3
metric rank=0, repeat=3, fold=4
metric rank=0, repeat=3, fold=5
metric rank=0, repeat=4, fold=1
metric rank=0, repeat=4, fold=2
metric rank=0, repeat=4, fold=3
metric rank=0, repeat=4, fold=4
metric rank=0, repeat=4, fold=5
metric rank=0, repeat=5, fold=1
metric rank=0, repeat=5, fold=2
metric rank=0, repeat=5, fold=3
metric rank=0, repeat=5, fold=4
metric rank=0, repeat=5, fold=5
metric rank=4, repeat=1, fold=1
metric rank=4, repeat=1, fold=2
metric rank=4, repeat=1, fold=3
metric rank=4, repeat=1, fold=4
metric rank=4, repeat=1, fold=5
metric rank=4, repeat=2, fold=1
metric r

,experiment,rank,CV_mean,CV_std,CV_median,CV_min,CV_max,delta_vs_raw,wins_vs_raw,losses_vs_raw,ties_vs_raw,wilcoxon_p,validation_auc,prediction_corr_with_raw,prediction_corr_with_current,auc_per_fold,mean_early_stop_epochs,external_refit_epochs
0,Diagonal monotone metric,0,0.859173,0.014181,0.859552,0.831743,0.880527,-0.070126,0,25,0,5.960464e-08,0.860702,0.855605,0.823146,"[0.8317428823381204, 0.8720867976820359, 0.855...",400.0,400
1,Low-rank monotone metric r=4,4,0.849354,0.015127,0.849364,0.817854,0.874323,-0.079945,0,25,0,5.960464e-08,0.849107,0.828272,0.796182,"[0.8178539934492315, 0.8589695137314186, 0.845...",400.0,400
2,Low-rank monotone metric r=8,8,0.847995,0.015222,0.848151,0.815697,0.873394,-0.081304,0,25,0,5.960464e-08,0.847270,0.824417,0.792421,"[0.8156966490299824, 0.8569223985890654, 0.844...",400.0,400
3,Low-rank monotone metric r=16,16,0.847135,0.015236,0.847285,0.814437,0.872654,-0.082164,0,25,0,5.960464e-08,0.846033,0.821979,0.790043,"[0.8144368858654573, 0.8555524061476443, 0.843...",400.0,400
4,Low-rank monotone metric r=32,32,0.846681,0.015263,0.846529,0.813697,0.872339,-0.082618,0,25,0,5.960464e-08,0.845306,0.820566,0.788537,"[0.8136967750062987, 0.8550327538422777, 0.843...",400.0,400


### Phase C — permutation and block/group sorting diagnostics


In [27]:
if RUN_STRUCTURE_PHASE_C:
    sorted_delta = structural_records['B. Sorted only']['auc_per_fold'].mean() - raw_aucs.mean()
    if sorted_delta <= 0 and not FORCE_STRUCTURE_PHASE_C:
        print('Sorted has no positive repeated-CV delta; Phase C skipped.')
        print('Set FORCE_STRUCTURE_PHASE_C=True to run the diagnostic anyway.')
    else:
        # Sanity check: sorting is exactly invariant to independent row permutations.
        X_permuted, sorted_max_abs_difference = permutation_invariance_diagnostic(X_train, SEED)
        print('max |sorted(X) - sorted(permuted X)| =', sorted_max_abs_difference)
        permutation_cache = build_repeated_distance_cache(
            X_permuted, y_train, STRUCTURAL_SEEDS, N_SPLITS,
            representations=('raw', 'sorted'),
        )
        permuted_raw_auc = evaluate_kernel(
            permutation_cache, structural_records['A. Raw only']['params']
        )
        permuted_sorted_auc = evaluate_kernel(
            permutation_cache, structural_records['B. Sorted only']['params']
        )
        permutation_results = pd.DataFrame({
            'representation': ['raw_permuted', 'sorted_permuted'],
            'CV_mean': [permuted_raw_auc.mean(), permuted_sorted_auc.mean()],
            'CV_std': [permuted_raw_auc.std(), permuted_sorted_auc.std()],
        })
        display(permutation_results)

        block_cache = build_repeated_distance_cache(
            X_train, y_train, STRUCTURAL_SEEDS, N_SPLITS,
            representations=('sorted', 'block2', 'block4', 'block8'),
        )
        block_rows = []
        for space in ('sorted', 'block2', 'block4', 'block8'):
            study, params, aucs, oof = optimize_structural_kernel(
                block_cache, (space,), 'single', SINGLE_SPACE_TRIALS, SEED,
            )
            val_auc, _ = external_validation_kernel(X_train, y_train, X_val, y_val, params)
            block_rows.append({
                'representation': space, 'CV_mean': aucs.mean(), 'CV_std': aucs.std(),
                'mean_delta_vs_global_sort': aucs.mean() - structural_records['B. Sorted only']['auc_per_fold'].mean(),
                'validation_auc': val_auc, 'params': params, 'auc_per_fold': aucs.tolist(),
            })
        block_sort_summary = pd.DataFrame(block_rows).sort_values('CV_mean', ascending=False)
        display(block_sort_summary.reset_index(drop=True))

        # Target-free grouping is only opened after a material fixed-block gain.
        if block_sort_summary.CV_mean.max() >= structural_records['B. Sorted only']['auc_per_fold'].mean() + 0.002:
            cluster_cache = build_repeated_distance_cache(
                X_train, y_train, STRUCTURAL_SEEDS, N_SPLITS,
                representations=('cluster4', 'cluster8', 'cluster16'),
            )
            cluster_rows = []
            for space in ('cluster4', 'cluster8', 'cluster16'):
                _, params, aucs, _ = optimize_structural_kernel(
                    cluster_cache, (space,), 'single', SINGLE_SPACE_TRIALS, SEED,
                )
                val_auc, _ = external_validation_kernel(X_train, y_train, X_val, y_val, params)
                cluster_rows.append({
                    'representation': space, 'CV_mean': aucs.mean(), 'CV_std': aucs.std(),
                    'validation_auc': val_auc, 'params': params,
                    'auc_per_fold': aucs.tolist(),
                })
            cluster_sort_summary = pd.DataFrame(cluster_rows).sort_values('CV_mean', ascending=False)
            display(cluster_sort_summary.reset_index(drop=True))


Sorted has no positive repeated-CV delta; Phase C skipped.
Set FORCE_STRUCTURE_PHASE_C=True to run the diagnostic anyway.


### Phase D — optional Gaussian copula diagnostics


In [28]:
if RUN_COPULA_PHASE_D:
    copula_cv = repeated_copula_cv(
        X_train, y_train, STRUCTURAL_SEEDS, N_SPLITS, n_bins=8, alpha=5.0,
    )
    copula_full = GaussianCopulaFeatures(n_bins=8, alpha=5.0, n_splits=5, seed=SEED)
    copula_full.fit(np.log1p(X_train.to_numpy()), y_train)
    copula_validation = copula_full.transform(np.log1p(X_val.to_numpy()))
    copula_rows = []
    for name, result in copula_cv.items():
        diag = paired_diagnostics(result['aucs'], raw_aucs)
        copula_rows.append({
            'experiment': name, 'CV_mean': result['aucs'].mean(),
            'CV_std': result['aucs'].std(), 'CV_median': np.median(result['aucs']),
            'delta_vs_raw': diag['mean_delta'], 'wins_vs_raw': diag['wins'],
            'losses_vs_raw': diag['losses'], 'wilcoxon_p': diag['wilcoxon_p'],
            'validation_auc': roc_auc_score(y_val, copula_validation[name]),
            'prediction_corr_with_raw': spearmanr(result['oof'].ravel(), raw_oof.ravel()).statistic,
            'auc_per_fold': result['aucs'].tolist(),
        })
    copula_summary = pd.DataFrame(copula_rows).sort_values('CV_mean', ascending=False)
    display(copula_summary.reset_index(drop=True))


copula repeat=1, fold=1
copula repeat=1, fold=2
copula repeat=1, fold=3
copula repeat=1, fold=4
copula repeat=1, fold=5
copula repeat=2, fold=1
copula repeat=2, fold=2
copula repeat=2, fold=3
copula repeat=2, fold=4
copula repeat=2, fold=5
copula repeat=3, fold=1
copula repeat=3, fold=2
copula repeat=3, fold=3
copula repeat=3, fold=4
copula repeat=3, fold=5
copula repeat=4, fold=1
copula repeat=4, fold=2
copula repeat=4, fold=3
copula repeat=4, fold=4
copula repeat=4, fold=5
copula repeat=5, fold=1
copula repeat=5, fold=2
copula repeat=5, fold=3
copula repeat=5, fold=4
copula repeat=5, fold=5


,experiment,CV_mean,CV_std,CV_median,delta_vs_raw,wins_vs_raw,losses_vs_raw,wilcoxon_p,validation_auc,prediction_corr_with_raw,auc_per_fold
0,marginal_llr,0.846952,0.016020,0.848529,-0.082347,0,25,1.227919e-05,0.840536,0.818259,"[0.8131613756613756, 0.8554894179894179, 0.846..."
1,marginal_plus_copula_llr,0.846848,0.015833,0.844687,-0.082451,0,25,5.960464e-08,0.874324,0.786082,"[0.8218065003779288, 0.8791572184429327, 0.849..."
2,copula_llr,0.467404,0.016772,0.466664,-0.461895,0,25,1.227919e-05,0.414911,-0.114495,"[0.47652116402116396, 0.48793776769967245, 0.4..."


### Recommendation summary — intentionally no submission


In [29]:
if RUN_STRUCTURAL_PHASE_A:
    best = structural_summary.sort_values('CV_mean', ascending=False).iloc[0]
    best_diag = paired_diagnostics(
        structural_records[best.experiment]['auc_per_fold'], raw_aucs,
    )
    print('STRUCTURAL STAGE SUMMARY')
    print('- best CV model:', best.experiment)
    print(f"- mean delta vs raw: {best_diag['mean_delta']:+.5f}")
    print(f"- wins/losses/ties: {best_diag['wins']}/{best_diag['losses']}/{best_diag['ties']}")
    print(f"- external validation delta: {best.validation_auc - raw_validation_auc:+.5f}")
    sorted_corr = spearmanr(
        structural_records['A. Raw only']['oof'].ravel(),
        structural_records['B. Sorted only']['oof'].ravel(),
    ).statistic
    print(f'- sorted complementary signal: Spearman(raw, sorted)={sorted_corr:.4f}')
    if RUN_MONOTONE_PHASE_B:
        metric_best = metric_summary.iloc[0]
        print(f"- best monotone metric: {metric_best.experiment}; "
              f"delta={metric_best.delta_vs_raw:+.5f}; "
              f"corr_raw={metric_best.prediction_corr_with_raw:.4f}")
    else:
        print('- monotone metric: Phase B not run')
    print('- No submission was fitted or written. Choose manually after review.')


STRUCTURAL STAGE SUMMARY
- best CV model: Baseline Current Best
- mean delta vs raw: +0.00174
- wins/losses/ties: 18/7/0
- external validation delta: +0.00616
- sorted complementary signal: Spearman(raw, sorted)=0.7965
- best monotone metric: Diagonal monotone metric; delta=-0.07013; corr_raw=0.8556
- No submission was fitted or written. Choose manually after review.
